In [5]:
import requests
import pandas as pd


In [6]:
# Request data from OMNIWEB
DEFAULT_VARS = {
    "BX_GSM": {"id": 12, "title": "Bx, GSE/GSM, nT"},
    "BY_GSM": {"id": 15, "title": "BY, nT (GSM)"},
    "BZ_GSM": {"id": 16, "title": "BZ, nT (GSM)"},
    "V": {"id": 24, "title": "SW Plasma Speed, km/s"},
    "N": {"id": 23, "title": "SW Proton Density, N/cm^3"},
    "T": {"id": 22, "title": "SW Plasma Temperature, K"}
}

payload = {
    "activity": "retrieve",
    "res": "hour",
    "scale": "Linear",
    "spacecraft": "omni2",
    "start_date": 20260201,
    "end_date": 20260201,
    "table": 0,
}
for param in DEFAULT_VARS.values():
    payload.setdefault("vars", [])
    payload["vars"].append(str(param.get("id")))

response = requests.post("https://omniweb.gsfc.nasa.gov/cgi/nx1.cgi", data=payload, timeout=120)
response.raise_for_status()

omni_response_text = response.text

print(omni_response_text)
    

<HTML>
<HEAD><TITLE>OMNIWeb Results</TITLE></HEAD>
<BODY>
<center><font size=5 color=red>OMNIWeb Plus Browser Results </font></center><br>
<B>Listing for omni2 data from 20260201 to 20260201</B><hr><pre>Selected parameters:
 1 BX, nT (GSE, GSM)
 2 BY, nT (GSM)
 3 BZ, nT (GSM)
 4 SW Plasma Speed, km/s
 5 SW Proton Density, N/cm^3
 6 SW Plasma Temperature, K

YEAR DOY HR    1     2     3     4     5        6 
2026  32  0   4.1  -1.7   0.2  377.   1.8   30691.
2026  32  1   4.2  -1.0   0.8  378.   3.2   56843.
2026  32  2   4.4  -1.0   0.9  378.   3.5   62815.
2026  32  3   4.9  -1.3   0.5  364.   3.9   51086.
2026  32  4   1.1  -2.7  -1.6  362.   4.4   40631.
2026  32  5  -3.8  -1.9  -2.5  376.   3.7   20212.
2026  32  6   1.4  -2.4  -1.2  367.   5.1   37403.
2026  32  7   4.4  -2.5  -0.3  348.   4.1   37723.
2026  32  8   4.9  -2.1   0.4  342.   3.8   38296.
2026  32  9   4.2  -1.1   0.6  343.   3.9   39370.
2026  32 10   0.9  -4.7  -1.8  347.   3.8   24372.
2026  32 11   4.0  -3.6  -0.

In [7]:
def _parse_omni_text(text: str) -> pd.DataFrame:
    rows: list[list[str]] = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split()
        if len(parts) < 3:
            continue
        if (
            parts[0].isdigit()
            and len(parts[0]) == 4 # Year column in text
            and parts[1].isdigit() # DOY (day of year) column in text
            and 1 <= len(parts[1]) <= 3
            and parts[2].isdigit() # HR (hour) column in text
            and 0 <= int(parts[2]) <= 23
        ):
            rows.append(parts)

    if not rows:
        raise RuntimeError("No OMNI records parsed; inspect response format or variable ids.")

    cols = ["year", "doy", "hr", *DEFAULT_VARS.keys()]
    trimmed = [r[: len(cols)] for r in rows if len(r) >= len(cols)]
    df = pd.DataFrame(trimmed, columns=cols)

    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    ts = (
        pd.to_datetime(
            df["year"].astype(int).astype(str)
            + df["doy"].astype(int).astype(str).str.zfill(3),
            format="%Y%j",
            utc=True,
        )
        + pd.to_timedelta(df["hr"].astype(int), unit="h")
    )
    df["timestamp"] = ts
    df = df.drop(columns=["year", "doy", "hr"])

    # OMNI missing observations are large numbers like 9999.9 or 9999.99
    for c in DEFAULT_VARS.keys():
        mask = (df[c].abs() >= 9_999) & (df[c].abs() <= 10_000)
        df.loc[mask, c] = pd.NA

    return df[["timestamp", "BX_GSM", "BY_GSM", "BZ_GSM", "V", "N", "T"]]

print(_parse_omni_text(omni_response_text))

                   timestamp  BX_GSM  BY_GSM  BZ_GSM      V    N        T
0  2026-02-01 00:00:00+00:00     4.1    -1.7     0.2  377.0  1.8  30691.0
1  2026-02-01 01:00:00+00:00     4.2    -1.0     0.8  378.0  3.2  56843.0
2  2026-02-01 02:00:00+00:00     4.4    -1.0     0.9  378.0  3.5  62815.0
3  2026-02-01 03:00:00+00:00     4.9    -1.3     0.5  364.0  3.9  51086.0
4  2026-02-01 04:00:00+00:00     1.1    -2.7    -1.6  362.0  4.4  40631.0
5  2026-02-01 05:00:00+00:00    -3.8    -1.9    -2.5  376.0  3.7  20212.0
6  2026-02-01 06:00:00+00:00     1.4    -2.4    -1.2  367.0  5.1  37403.0
7  2026-02-01 07:00:00+00:00     4.4    -2.5    -0.3  348.0  4.1  37723.0
8  2026-02-01 08:00:00+00:00     4.9    -2.1     0.4  342.0  3.8  38296.0
9  2026-02-01 09:00:00+00:00     4.2    -1.1     0.6  343.0  3.9  39370.0
10 2026-02-01 10:00:00+00:00     0.9    -4.7    -1.8  347.0  3.8  24372.0
11 2026-02-01 11:00:00+00:00     4.0    -3.6    -0.4  335.0  4.4  32582.0
12 2026-02-01 12:00:00+00:00     3.6  